In [1]:
import torch
import numpy as np
from utils import *

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float
print("device:", device)
print("torch:", torch.__version__)
print("imports OK")

device: cpu
torch: 2.8.0+cpu
imports OK


In [2]:
x_in  = 10
H     = 30
x_out = 30

s1_tr, s1_te, s2_tr, s2_te = sample_hdgm_semi_t2(
    n_train=100, n_test=100, d=x_in, kk=0, level="hard")

print("s1_tr shape:", s1_tr.shape)
print("s2_tr shape:", s2_tr.shape)
print("P mean (first 2 dims):", s1_tr[:, :2].mean(axis=0).round(3))
print("Q mean (first 2 dims):", s2_tr[:, :2].mean(axis=0).round(3))
print("means should look nearly identical")

s1_tr shape: (200, 10)
s2_tr shape: (200, 10)
P mean (first 2 dims): [0.302 0.2  ]
Q mean (first 2 dims): [0.279 0.287]
means should look nearly identical


In [3]:
# Phase 1 — autoencoder on pooled unlabelled data
S_enc = MatConvert(
    np.concatenate([s1_tr, s1_te, s2_tr, s2_te]), device, dtype)

encoder = train_autoencoder(
    S_enc, epoch=100,
    x_in=x_in, H=H, x_out=x_out,
    batch_size=512, device=device, dtype=dtype, lr=0.002)

for p in encoder.parameters():
    p.requires_grad = False

print("Phase 1 done — encoder frozen")

# Phase 2 — classifier head on frozen encoder
S_tr = MatConvert(np.concatenate([s1_tr, s2_tr]), device, dtype)
y_tr = torch.cat([
    torch.zeros(s1_tr.shape[0]),
    torch.ones(s2_tr.shape[0])
]).to(device, dtype).long()

model_rl = ExtendedModel(encoder, H, x_out)
model_rl, w, b = C2ST_NN_fit(
    S_tr, y_tr, x_in, H, x_out,
    N_epoch=100, batch_size=512,
    device=device, dtype=dtype,
    model=model_rl, lr_c2st=0.002)

print("Phase 2 done — classifier trained")

# Phase 3 — permutation test on held-out split
S_te = MatConvert(np.concatenate([s1_te, s2_te]), device, dtype)

h, threshold, stat = TST_C2ST(
    S_te, N1=s1_te.shape[0],
    N_per=100, alpha=0.05,
    model_C2ST=model_rl, w_C2ST=w, b_C2ST=b)

print(f"\nreject H0: {bool(h)}")
print(f"stat:      {float(stat):.4f}")
print(f"threshold: {float(threshold):.4f}")
print("\nPipeline working correctly if no errors above.")

Epoch [60/100], Loss: 0.7393
Phase 1 done — encoder frozen
Phase 2 done — classifier trained

reject H0: False
stat:      0.0900
threshold: 0.0900

Pipeline working correctly if no errors above.


In [5]:
import json, subprocess, time

meta = {
    "commit"  : subprocess.check_output(
                    ["git", "rev-parse", "--short", "HEAD"],
                    cwd=r"C:\Users\midhu\Documents\GitHub\A-Unified-Data-Representation-Learning-for-Non-parametric-Two-sample-Testing"
                ).decode().strip(),
    "method"  : "RL-C2ST",
    "dataset" : "HDGM-D",
    "d"       : x_in,
    "level"   : "hard",
    "n_train" : 100,
    "epochs"  : {"ae": 100, "c2st": 100},
    "lr"      : {"ae": 0.002, "c2st": 0.002},
    "result"  : {"h": int(h), "stat": float(stat), "threshold": float(threshold)},
    "ts"      : time.strftime("%Y-%m-%d %H:%M"),
}

with open("result/smoke_test.json", "w") as f:
    json.dump(meta, f, indent=2)

print("saved to result/smoke_test.json")
print(json.dumps(meta, indent=2))

saved to result/smoke_test.json
{
  "commit": "adb2920",
  "method": "RL-C2ST",
  "dataset": "HDGM-D",
  "d": 10,
  "level": "hard",
  "n_train": 100,
  "epochs": {
    "ae": 100,
    "c2st": 100
  },
  "lr": {
    "ae": 0.002,
    "c2st": 0.002
  },
  "result": {
    "h": 0,
    "stat": 0.0899999737739563,
    "threshold": 0.0899999737739563
  },
  "ts": "2026-07-07 14:11"
}
